In [9]:
from pathlib import Path 
import sys 

import numpy as np 
import pandas as pd 
from PIL import Image 

PROJECT_ROOT = Path.cwd() 

if PROJECT_ROOT.name == "notebooks": 
    PROJECT_ROOT = PROJECT_ROOT.parent 

if not (PROJECT_ROOT / "src" / "embedder.py").exists(): 
    raise FileNotFoundError(
        f"Could not find src/embedder.py from {PROJECT_ROOT}"
    )
sys.path.insert(0, str(PROJECT_ROOT)) 
from src.embedder import Embedder

/Users/aob/AOB/CS/Projects/clip-semantic-search/.venv/lib/python3.13/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
W1101 15:17:16.509000 65111 .venv/lib/python3.13/site-packages/torch/distributed/elastic/multiprocessing/redirects.py:35] NOTE: Redirects are currently not supported in MacOs.


In [10]:
manifest_path = PROJECT_ROOT / "data" / "metadata" / "abo_subset.csv" 

manifest = pd.read_csv(manifest_path) 

print(f"Manifest path: {manifest_path}") 
print(f"Number of records: {len(manifest)}")
print(manifest.head(3))

Manifest path: /Users/aob/AOB/CS/Projects/clip-semantic-search/data/metadata/abo_subset.csv
Number of records: 1000
      item_id domain_name     image_id  \
0  B084978DK5   amazon.in  31Ca8NxZktL   
1  B08562DD6Z   amazon.in  31Dzqdw8rwL   
2  B0093VF79G  amazon.com  31NJk+Ie1JL   

                                          image_path  \
0  data/ABO/abo-images-small/images/small/12/1229...   
1  data/ABO/abo-images-small/images/small/b6/b663...   
2  data/ABO/abo-images-small/images/small/04/0401...   

                                               title         product_type  \
0  Amazon Brand - Solimo Soft & Flexible Back Pho...  CELLULAR_PHONE_CASE   
1  Amazon Brand - Solimo Soft & Flexible Back Pho...  CELLULAR_PHONE_CASE   
2  AmazonBasics Clear Cover Case with Screen Prot...  CELLULAR_PHONE_CASE   

                   brand        color         style  
0  Amazon Brand - Solimo  Transparent  professional  
1  Amazon Brand - Solimo  Transparent  professional  
2           AmazonB

In [11]:
required_columns = {
    "item_id", 
    "image_id", 
    "image_path", 
    "title", 
    "product_type", 
}

missing_columns = required_columns - set(manifest.columns) 

if missing_columns: 
    raise ValueError(f"Missing columns: {missing_columns}") 

if len(manifest) != 1000: 
    raise ValueError(f"Expected 1000 records, found {len(manifest)}")

print("Manifest structure and record count are valid.")

Manifest structure and record count are valid.


In [13]:
image_paths = [
    PROJECT_ROOT / relative_path 
    for relative_path in manifest["image_path"]
]

missing_paths = [
    path for path in image_paths 
    if not path.exists() 
]

print(f"Total image paths: {len(image_paths)}")
print(f"Missing image paths: {len(missing_paths)}") 

if missing_paths: 
    print("First missing path:", missing_paths[0]) 
    raise FileNotFoundError("One or more manifest image paths are missing.")

print("All manifest image files exist.")

Total image paths: 1000
Missing image paths: 0
All manifest image files exist.


In [14]:
# Instantiate the embedder 
embedder = Embedder() 

print(f"Device: {embedder.device}") 
print(f"Project dimension: {embedder.model.config.projection_dim}") 

Loading weights: 100%|██████████| 398/398 [00:00<00:00, 73926.44it/s]


Device: mps
Project dimension: 512


In [15]:
batch_size = 16

In [16]:
test_paths = image_paths[:batch_size] 

test_images = [] 

for image_path in test_paths: 
    with Image.open(image_path) as image: 
        test_images.append(image.convert("RGB"))

print(f"Loaded images: {len(test_images)}")
print(f"First image size: {test_images[0].size}") 

Loaded images: 16
First image size: (193, 256)


In [17]:
test_embeddings = embedder.embed_images(test_images) 

print(f"Embedding shape: {test_embeddings.shape}") 
print(f"Embedding dtype: {test_embeddings.dtype}") 
print("Embedding norms:", np.linalg.norm(test_embeddings, axis=1)) 

Embedding shape: (16, 512)
Embedding dtype: float32
Embedding norms: [0.99999994 1.         0.99999994 1.         1.         0.99999994
 1.         1.         1.         1.         1.         1.
 1.         0.99999994 1.         0.99999994]


In [18]:
expected_shape = (batch_size, embedder.model.config.projection_dim) 

if test_embeddings.shape != expected_shape: 
    raise ValueError(
        f"Expected shape {expected_shape}" 
        f"received {test_embeddings.shape}" 
    )

if not np.isfinite(test_embeddings).all(): 
    raise ValueError("Embeddings contain NaN or infinite values.") 

if not np.allclose(
    np.linalg.norm(test_embeddings, axis=1), 
    1.0, 
    atol=1e-5, 
): 
    raise ValueError("Embeddings are not normalized.") 

print("The first image batch passed validation.")

The first image batch passed validation.


In [20]:
all_embeddings = [] 

for start_index in range(0, len(image_paths), batch_size): 
    end_index = min(start_index + batch_size, len(image_paths)) 
    batch_paths = image_paths[start_index:end_index] 

    batch_images = [] 

    for image_path in batch_paths: 
        try: 
            with Image.open(image_path) as image: 
                batch_images.append(image.convert("RGB"))
        except OSError as error: 
            raise RuntimeError(
                f"Could not read image: {image_path}"
            ) from error 

    batch_embeddings = embedder.embed_images(batch_images) 
    all_embeddings.append(batch_embeddings)

    print(
        f"Processed images {start_index} "
        f"through {end_index - 1}"
    )

Processed images 0 through 15
Processed images 16 through 31
Processed images 32 through 47
Processed images 48 through 63
Processed images 64 through 79
Processed images 80 through 95
Processed images 96 through 111
Processed images 112 through 127
Processed images 128 through 143
Processed images 144 through 159
Processed images 160 through 175
Processed images 176 through 191
Processed images 192 through 207
Processed images 208 through 223
Processed images 224 through 239
Processed images 240 through 255
Processed images 256 through 271
Processed images 272 through 287
Processed images 288 through 303
Processed images 304 through 319
Processed images 320 through 335
Processed images 336 through 351
Processed images 352 through 367
Processed images 368 through 383
Processed images 384 through 399
Processed images 400 through 415
Processed images 416 through 431
Processed images 432 through 447
Processed images 448 through 463
Processed images 464 through 479
Processed images 480 thr

In [21]:
image_embedings = np.vstack(all_embeddings) 

print("Final embedding shape:", image_embedings.shape) 
print("Embedding dtype:", image_embedings.dtype) 

Final embedding shape: (1000, 512)
Embedding dtype: float32


In [22]:
if image_embedings.shape != (len(manifest), embedder.model.config.projection_dim): 
    raise ValueError(f"Unexpected shape: {image_embedings.shape}") 

if not np.isfinite(image_embedings).all(): 
    raise ValueError("Embeddings contain NaN or infinite values.") 

embedding_norms = np.linalg.norm(image_embedings, axis=1) 

if not np.allclose(embedding_norms, 1.0, atol=1e-5): 
    raise ValueError("Some embeddings are not normalized.")

print("All 1,000 image embeddings passed validation.")

All 1,000 image embeddings passed validation.


In [23]:
embeddings_dir = PROJECT_ROOT / "data" / "embeddings"
embeddings_dir.mkdir(parents=True, exist_ok=True) 

embeddings_path = embeddings_dir / "abo_image_embeddings.npy" 
metadata_path = embeddings_dir / "abo_image_metadata.csv" 

np.save(embeddings_path, image_embedings) 
manifest.to_csv(metadata_path, index=False) 

print(f"Saved embeddings to: {embeddings_path}")
print(f"Saved metadata to: {metadata_path}")

Saved embeddings to: /Users/aob/AOB/CS/Projects/clip-semantic-search/data/embeddings/abo_image_embeddings.npy
Saved metadata to: /Users/aob/AOB/CS/Projects/clip-semantic-search/data/embeddings/abo_image_metadata.csv


In [24]:
reloaded_embeddings = np.load(embeddings_path) 
reloaded_metadata = pd.read_csv(metadata_path) 

print(reloaded_embeddings.shape) 
print(len(reloaded_metadata)) 

if reloaded_embeddings.shape[0] != len(reloaded_metadata): 
    raise ValueError("Embedding and metadata row counts do not match.") 

print("Saved files reload correctly.")

(1000, 512)
1000
Saved files reload correctly.
